1. Standardize dataset
2. Obtain covariance matrix

3. Compute Eigenvalues and Eigenvectors
4. Sort the Eigenvalues in descending order
5. Apply the same order to Eigenvectors as well
6. Choose threshold k -> This threshold k determines how many principal components to keep, often based on the amount of total variance you want to retain in your data. For example, you might choose k so that the first k principal components together explain at least 95% of the variance.

7. Select k Eigenvectors with k largest Eigenvalues
8. Extract k dimensional feature vectors

In [24]:
import plotly.express as px
import pandas as pd
import numpy as np

class PCA:
    def __init__(self, n_components=5, random_state=42, standardize=True):
        self.n_components = n_components
        self.random_state = random_state
        self.standardize = standardize
    
    def fit(self,X):
        # 1. create a copy of X as X_copy
        self.X_copy = X.copy()

        # 2. standardize is needed, basically (X - mean) / std
        if self.standardize:
            mean = np.mean(self.X_copy, axis=0) # axis=0 -> since we wanna average all data, not over features
            std = np.std(self.X_copy, axis=0) + 1e-9 # just for epsilon so it does not get to zero
            self.X_copy = (self.X_copy - mean) / std 

        # 3. set up random state
        if self.random_state:
            np.random.seed(self.random_state)
        
        # 4. get covariance matrix
        cov_mat = np.cov(self.X_copy.T) 
        # Q: why do we need to transpose? initially row=data, col=features
        # A: We transpose (self.X_copy.T) because np.cov expects features to be rows and samples to be columns; by transposing, we ensure the covariance matrix reflects feature relationships, not sample relationships.

        # 5. get eigenvalues and eigenvectors using np.linalg.eig(cov_mat) -> return val, vec simultaneoursly
        val,vec = np.linalg.eig(cov_mat)

        # 6. sort in descending order , keep the indices
        idxs = np.argsort(val)[::-1]
        self.val = val[idxs]
        self.vec = vec[:,idxs]
        # Q: why we need to add ":", so over the 2nd axes?
        # A: val is scalar since its eigenvalue, eigenvector we need to keep the row and reorder the column, should reorder the columns: vec[:, idxs]. This way, each eigenvector stays aligned with its eigenvalue.
        # why: remember above cov_mat = np.cov(self.X_copy.T) , we already transpose since our column now becomes the "data-point", so we need to reorder based on the column, not row anymore

        # absolute value indexing mode
        if self.n_components >= 1:
            # 7. based on the sorted, get the number of components based on self.n_components
            top_n_vec = self.vec[:,:self.n_components]
            top_n_val = self.val[:self.n_components]
            total_val = np.sum(self.val)

            # 8. get explained variance ratio for each selected component
            self.explained_var_ratio = top_n_val / total_val
            # 9. cumulatively sum the explained variance ratios
            self.cum_explained_var = np.cumsum(self.explained_var_ratio)

            # 10. create the projection matrix
            self.projection_matrix = top_n_vec

        # ratio mode: n_components in (0, 1)
        elif 0 < self.n_components < 1:
            # Compute the explained variance ratio for all components
            var_ratio_all = self.val / np.sum(self.val)
            cum_var_ratio = np.cumsum(var_ratio_all)
            # Find the minimum number of components needed to reach the desired variance threshold
            self.n_components = np.searchsorted(cum_var_ratio, self.n_components) + 1
            top_n_vec = self.vec[:, :self.n_components]
            top_n_val = self.val[:self.n_components]
            self.explained_var_ratio = top_n_val / np.sum(self.val)
            self.cum_explained_var = np.cumsum(self.explained_var_ratio)
            self.projection_matrix = top_n_vec

    def fit_transform(self, X, y):
        # fit so we can obtain both X_copy and the projection matrix
        self.fit(X)
        # calculate all X with the projection matrix
        self.components_ = np.dot(self.X_copy, self.projection_matrix)
        # then, for each obtained component, it supposed to be the n_components amount of data / PC{i}
        self.df = pd.DataFrame(data=self.components_, columns=['PC{}'.format(i+1) for i in range(self.n_components)])
        self.df['label'] = y
        # we can just return the components for just getting the raw transformed data, the df is just for visualization purposes
        return self.components_
    
    def transform(self,X,y):
        self.components_ = np.dot(X, self.projection_matrix)
        self.df = pd.DataFrame(data=self.components_, columns=['PC{}'.format(i+1) for i in range(self.n_components)])
        self.df['label'] = y
        return self.components_

    def pca_plot2d(self):
        fig = px.scatter(self.components_, x=0, y=1, color=self.df.label,labels={'0': 'PC 1', '1': 'PC 2'})
        fig.show()

    def pca_plot3d(self):
        fig = px.scatter_3d(self.components_, x=0, y=1,z=2, color=self.df.label,labels={'0': 'PC 1', '1': 'PC 2'})
        fig.show()

Let's try

In [13]:
import pandas as pd
import numpy as np
import random
from sklearn.datasets import load_breast_cancer
import plotly.express as px

data = load_breast_cancer(as_frame=True)
X,y,df_bre = data.data,data.target,data.frame
diz_target = {0:'malignant',1:'benign'}
y = np.array([diz_target[y1] for y1 in y])
df_bre['target'] = df_bre['target'].apply(lambda x: diz_target[x])
df_bre.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,malignant
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,malignant
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,malignant
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,malignant
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,malignant


In [16]:
pca1 = PCA(n_components=4,random_state=True,standardize=False)
X_new = pca1.fit_transform(X,y)
print('4D feature subspace:\n',X_new)
print('\nVariance Explained Ratio:\n',pca1.explained_var_ratio)
print('\nCumulative Variance Explained:\n',pca1.cum_explained_var)

4D feature subspace:
 [[2260.01388629 -187.96030122   17.91296119  -77.26294188]
 [2368.99375578  121.58742426  -66.05997067  -50.68968334]
 [2095.66520155  145.11398566  -32.37518942  -64.35162646]
 ...
 [1414.37306877  153.5107676   -41.10784362  -78.32284761]
 [2224.72942789  140.08646738  -50.40752386  -92.21184801]
 [ 328.34369071   17.31413605   -6.77640455  -66.00371802]]

Variance Explained Ratio:
 [9.82044672e-01 1.61764899e-02 1.55751075e-03 1.20931964e-04]

Cumulative Variance Explained:
 [0.98204467 0.99822116 0.99977867 0.9998996 ]


In [17]:
pca1.pca_plot2d()

In [18]:
pca1.pca_plot3d()

with standardization

In [20]:
pca1 = PCA(n_components=4,random_state=True,standardize=True)
X_new = pca1.fit_transform(X,y)
print('4D feature subspace:\n',X_new)
print('\nCumulative Variance Explained:\n',pca1.cum_explained_var)

4D feature subspace:
 [[ 9.19283654  1.94858359 -1.12316569  3.63373089]
 [ 2.38780206 -3.76817152 -0.52929328  1.11826383]
 [ 5.73389623 -1.07517337 -0.55174775  0.91208261]
 ...
 [ 1.25617939 -1.9022966   0.5627303  -2.08922689]
 [10.37479381  1.67201097 -1.87702873 -2.35603113]
 [-5.47524306 -0.67063714  1.49044307 -2.29915699]]

Cumulative Variance Explained:
 [0.44272028 0.63243209 0.72636372 0.79238507]


In [22]:
pca1.pca_plot2d()

In [23]:
pca1.pca_plot3d()